In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
expected_capability_version = "1.0.0"
pipeline_run_id = ""
expected_installed_capabilities =  {}
prefix = ""
deployment_environment = ""

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 21, Finished, Available, Finished)

In [ ]:
import io
import requests
import unittest
import logging
import sempy.fabric as fabric
import json
import pandas as pd
import xmlrunner
from datetime import datetime, timedelta

EXPECTED_OMOP_TABLES = [
    'care_site',
    'cdm_source',
    'cohort',
    'cohort_definition',
    'concept',
    'concept_ancestor',
    'concept_class',
    'concept_relationship',
    'concept_synonym',
    'condition_era',
    'condition_occurrence',
    'cost',
    'death',
    'device_exposure',
    'domain',
    'dose_era',
    'drug_era',
    'drug_exposure',
    'drug_strength',
    'episode',
    'episode_event',
    'fact_relationship',
    'fhir_system_to_omop_vocab_mapping',
    'image_occurrence',
    'location',
    'measurement',
    'metadata',
    'note',
    'note_nlp',
    'observation',
    'observation_period',
    'payer_plan_period',
    'person',
    'procedure_occurrence',
    'provider',
    'relationship',
    'source_to_concept_map',
    'specimen',
    'visit_detail',
    'visit_occurrence',
    'vocabulary'
]


PBI_GLOBAL_SERVICE_ENDPOINTS = {
    "public": "https://api.powerbi.com/",
    "fairfax": "https://api.powerbigov.us",
    "mooncake": "https://api.powerbi.cn",
    "blackforest": "https://app.powerbi.de",
    "msit": "https://api.powerbi.com/",
    "prod": "https://api.powerbi.com/",
    "int3": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
    "dxt": "https://powerbistagingapi.analysis.windows.net/",
    "edog": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
    "dev": "https://onebox-redirect.analysis.windows-int.net/",
    "console": "http://localhost:5001/",
    "daily": "https://dailyapi.powerbi.com/",
}

DEFAULT_GLOBAL_SERVICE_ENDPOINT = "https://api.powerbi.com/"
FETCH_CLUSTER_DETAIL_URI = "powerbi/globalservice/v201606/clusterDetails"

class OmopAnalyticsDeploymentTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, env = "msit", workspace_id = None, bronze_lakehouse_id = None, expected_installed_capabilities=[], prefix="healthcare1"):
        super().__init__(methodName)
        logging.basicConfig()
        self.client = fabric.FabricRestClient()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.env = env
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.expected_capability_version = expected_capability_version
        self.expected_installed_capabilities = expected_installed_capabilities
        self.prefix = prefix

    def setUp(self):
        self.token = mssparkutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
        self.runtime_context = mssparkutils.runtime.context
        self.workspace_id = self.runtime_context["currentWorkspaceId"]
        self.capacity_id = self.client.get(f"v1/workspaces/{self.workspace_id}").json()["capacityId"]
        self.solution_id = self.client.get(f"v1/workspaces/{self.workspace_id}/items?type=Healthcaredatasolution").json()["value"][0]["id"]
        self.installed_capabilities = self.get_installed_capabilities(self.env, self.capacity_id, self.workspace_id, self.solution_id, self.token)
        self.available_capabilities = self.get_all_capabilities(self.env, self.capacity_id, self.workspace_id, self.solution_id, self.token, "1.0.0")

    # Helper methods
    def get_shared_host(self, env: str, token: str):
        
        url = PBI_GLOBAL_SERVICE_ENDPOINTS.get(env, DEFAULT_GLOBAL_SERVICE_ENDPOINT) + FETCH_CLUSTER_DETAIL_URI
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
        resp = requests.get(url, headers=headers)
        resp_body = json.loads(resp.content)
        return resp_body["clusterUrl"]

    def get_request_info(self, env, capacity_id, workspace_id, solution_id, shared_host = None, token = None):
        mwc_token_details = self.get_hds_mwc_token_details(env, capacity_id, workspace_id, workload_type="dmh", shared_host=shared_host, token=token)
        
        mwc_token = mwc_token_details["Token"]
        target_uri = mwc_token_details["TargetUriHost"]
        
        return[
            target_uri, 
            {
                'Authorization': f'MwcToken {mwc_token}',
                'Content-Type': 'application/json',
                'x-ms-workload-resource-moniker': solution_id
            }
        ]

    def get_hds_mwc_token_details(self, env: str, capacity_id: str, workspace_id: str, workload_type: str = "dmh", shared_host = None, token: str = ""):

        if shared_host is None:
            shared_host = self.get_shared_host(env, token)

        payload = {
            "capacityObjectId": capacity_id,
            "workspaceObjectId": workspace_id,
            "workloadType": workload_type,
        }

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        url = f"{shared_host}/metadata/v201606/generatemwctokenv2"
        response = requests.post(url=url, data=json.dumps(payload), headers=headers)
        mwc_token_details = json.loads(response.content)
        return mwc_token_details

    def get_installed_capabilities(self, env, capacity_id: str, workspace_id: str, solution_id: str, token):
        
        shared_host = self.get_shared_host(env, token)
        [target_uri, headers] = self.get_request_info(env, capacity_id, workspace_id, solution_id, shared_host, token)
        endpoint = "https://{}/webapi/capacities/{}/workloads/dmh/DMHService/automatic/artifacts/{}/capabilities"
        url = endpoint.format(target_uri, capacity_id, solution_id)
        response = requests.get(url=url, headers=headers)
        payload = response.json()
        # print(payload)
        return payload

    def get_all_capabilities(self, env, capacity_id, workspace_id, solution_id, token, version = "1.0.0"):

        shared_host = self.get_shared_host(env, token)
        [target_uri, headers] = self.get_request_info(env, capacity_id, workspace_id, solution_id, shared_host, token)
        endpoint = "https://{}/webapi/capacities/{}/workloads/dmh/DMHService/automatic/capabilities?artifactVersion={}"
        url = endpoint.format(target_uri, capacity_id, version)
        response = requests.get(url=url, headers=headers)
        return response.json()

    def get_workspace_items_by_type(self, workspace_id, item_type):
        client = fabric.FabricRestClient()

        response = client.get(f"v1/workspaces/{workspace_id}/items?type={item_type}")
        items = response.json()['value']

        item_dict = {}
        for item in items:
            item_dict[item['displayName']] = item
        
        return item_dict

    # Tests
    def test_installed_capabilities_match_expected(self):
        installed_capabilities_keys = [c["name"] for c in self.installed_capabilities]
        for key in self.expected_installed_capabilities:
            self.assertIn(key, installed_capabilities_keys)
    def test_capabilities_deployed_successfully(self):
        capability_provisioning_states = [c["provisionState"] for c in self.installed_capabilities]
        for capability_provisioning_state in capability_provisioning_states:
            self.assertEqual("Active", capability_provisioning_state)

    def test_deployed_capabilities_have_expected_version(self):
        print(self.installed_capabilities)

        capability_versions = { c["name"] : c["version"] for c in self.installed_capabilities }
        for expected_capability in self.expected_installed_capabilities.keys():
            self.assertTrue(expected_capability in capability_versions)
            self.assertEqual(self.expected_installed_capabilities[expected_capability], capability_versions[expected_capability])

    def test_omop_silver_gold_transformation_lakehouse_deployed_succesfully(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")
        self.assertIn(prefix + "_msft_gold_omop", deployed_lakehouses)

    def test_omop_notebooks_deployed_successfully(self):
        
        deployed_notebooks = self.get_workspace_items_by_type(self.workspace_id, "Notebook")

        self.assertTrue(prefix + "_msft_omop_drug_exposure_insights_sample" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_omop_drug_exposure_era_sample" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_omop_silver_gold_transformation" in deployed_notebooks)

    def test_dtt_configuration_files_present(self):

        expected_dtt_configuration_files = [
            "dbSemantics.json",
            "dbSemanticsConfig.json",
            "dbTargetSchema.json",
            "dbTargetSchemaConfig.json",
            "dmfAdapter.json",
            "dmfAdapterSchema.json"
        ]

        item_type = "HealthcareDataSolution"
        client = fabric.FabricRestClient()

        response = client.get(f"v1/workspaces/{self.workspace_id}/items?type={item_type}")
        solution_id = response.json()['value'][0]['id']

        dtt_configuration_file_infos = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{solution_id}/DMHConfiguration/_internal/fhir4/transformation/omop")
        
        actual_dtt_config_files = {}
        for dtt_config in dtt_configuration_file_infos:
            actual_dtt_config_files[dtt_config.path.split("/")[-1]] = dtt_config.size

        # Assert the expected files exist and are not empty
        for expected_dtt_file in expected_dtt_configuration_files:
            self.assertIn(expected_dtt_file, actual_dtt_config_files)
            self.assertTrue(actual_dtt_config_files[expected_dtt_file] > 0)

    def test_omop_analytics_datapipeline_deployed_successfully(self):

        deployed_datapipelines = self.get_workspace_items_by_type(self.workspace_id, "DataPipeline")
        self.assertTrue(prefix + "_msft_omop_analytics" in deployed_datapipelines)

    def test_environment_deployed_successfully(self):
        
        deployed_environments = self.get_workspace_items_by_type(self.workspace_id, "Environment")
        self.assertTrue(prefix + "_environment" in deployed_environments)

    def test_omop_vocab_reference_data_deployed(self):
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        bronze_lakehouse = None
        for lh in deployed_lakehouses.keys():
            if "bronze" in str(lh).lower():
                bronze_lakehouse = deployed_lakehouses[lh]
        
        bronze_lakehouse_id = bronze_lakehouse["id"]
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Files/ReferenceData/Clinical/OMOP/Vocab-HDS"))

    def test_validate_omop_tables_exist(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        omop_lakehouse = None
        print(deployed_lakehouses.keys())
        for lh in deployed_lakehouses.keys():
            if "omop" in str(lh).lower():
                omop_lakehouse = deployed_lakehouses[lh]

        self.assertTrue(omop_lakehouse is not None)

        omop_lakehouse_id = omop_lakehouse["id"]

        omop_lakehouse_path = f"abfss://{workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{omop_lakehouse_id}/Tables"
        print(omop_lakehouse_path)
        deployed_omop_tables = [str(p.path).split("/")[-1] for p in mssparkutils.fs.ls(omop_lakehouse_path)]

        self.assertEqual(len(EXPECTED_OMOP_TABLES), len(deployed_omop_tables))

        for table in EXPECTED_OMOP_TABLES:
            self.assertTrue(table in deployed_omop_tables)

    def test_validate_omop_activity_configured_in_deployment_parameters(self):

        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        for lh in deployed_lakehouses.keys():
            if "admin" in str(lh).lower():
                admin_lakehouse = deployed_lakehouses[lh]

        admin_lakehouse_id = admin_lakehouse["id"]
        deployment_params_file_path = f"abfss://{workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Files/system-configurations/deploymentParametersConfiguration.json"
        deployment_params_data = json.loads(spark.read.json(deployment_params_file_path, multiLine=True).toJSON().collect()[0])

        configured__omop_activity = None

        for activity in deployment_params_data["activities"].values():
            if "omop" in activity["name"]:
                configured__omop_activity = activity

        self.assertTrue("parameters" in configured__omop_activity)
        self.assertTrue("vocab_path" in configured__omop_activity["parameters"] and len(configured__omop_activity["parameters"]["vocab_path"]) > 0)
        self.assertTrue("vocab_checkpoint_path" in configured__omop_activity["parameters"] and len(configured__omop_activity["parameters"]["vocab_checkpoint_path"]) > 0)
        self.assertTrue("omop_config_path" in configured__omop_activity["parameters"] and len(configured__omop_activity["parameters"]["omop_config_path"]) > 0)

    def test_validate_omop_deployment_parameters_have_matching_artifacts(self):

        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        for lh in deployed_lakehouses.keys():
            if "admin" in str(lh).lower():
                admin_lakehouse = deployed_lakehouses[lh]

        admin_lakehouse_id = admin_lakehouse["id"]
        deployment_params_file_path = f"abfss://{workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Files/system-configurations/deploymentParametersConfiguration.json"
        deployment_params_data = json.loads(spark.read.json(deployment_params_file_path, multiLine=True).toJSON().collect()[0])

        configured_activities = {}
        for activity in deployment_params_data["activities"].keys():
            configured_activities[activity] = deployment_params_data["activities"][activity]['name']

        deployed_notebooks = self.get_workspace_items_by_type(self.workspace_id, "Notebook")

        deployed_notebooks_dict = {}
        for nb_name, nb_data in deployed_notebooks.items():
            deployed_notebooks_dict[nb_data["id"]] = nb_data["displayName"]

        print(deployed_notebooks_dict)

        for activity in configured_activities.keys():
            print(activity)
            self.assertIn(activity, deployed_notebooks_dict)
            self.assertTrue(configured_activities[activity] == deployed_notebooks_dict[activity])

def run_tests_and_write_output(spark):
    
    # Run the tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(OmopAnalyticsDeploymentTests)
    print([test.__str__().split(" ")[0] for test in suite])

    for test in suite:
        print(test.__str__().split(" ", maxsplit=1)[0])
        test.spark = spark
        test.env = deployment_environment
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
        test.expected_capability_version = expected_capability_version
        test.expected_installed_capabilities = expected_installed_capabilities
        test.prefix = prefix

    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream).run(suite)

    xml_output = write_stream.getvalue().decode('utf-8')

    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)

run_tests_and_write_output(spark)

StatementMeta(, 3308407d-eaeb-44ee-927a-435e6e433003, 21, Finished, Available, Finished)


Running tests...
----------------------------------------------------------------------
F...
FAIL [4.666s]: test_available_capabilities_match_expected (__main__.CapabilityDeploymentTests.test_available_capabilities_match_expected)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_14781/3932477369.py", line 156, in test_available_capabilities_match_expected
    self.assertTrue(False)
AssertionError: False is not true

----------------------------------------------------------------------
Ran 4 tests in 13.623s

FAILED (failures=1)

Generating XML reports...
